# CONUS HumanET × SIF × Drought Analysis

**HumanET** = OpenET − NLDAS Noah ET isolates anthropogenic water additions (primarily irrigation). This notebook extends the Iowa analysis to all of CONUS for 2015–2024.

**Prerequisites**: Run all download scripts (06, 16–19) then `00_align_spatial_data_conus.ipynb` before this notebook.

**Approach**: Continuous GRIDMET drought indices (SPI, SPEI, EDDI) are binned into USDM-equivalent categories for comparison with SIF anomalies across irrigated vs rainfed cropland.

In [ ]:
import sys, os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from datetime import date

_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

proc  = project_root / 'data' / 'processed' / 'conus'
raw   = project_root / 'data' / 'raw'
figs  = project_root / 'figures' / 'conus'
figs.mkdir(parents=True, exist_ok=True)

# Input subdirs
cdl_proc    = proc / 'cdl'
openet_proc = proc / 'openet'
nldas_proc  = proc / 'nldas'
ndvi_proc   = proc / 'ndvi'
sif_proc    = proc / 'sif'
usdm_proc   = proc / 'drought_usdm'
gmet_proc   = proc / 'drought_gridmet'

# CONUS NLDAS grid
CONUS_LON = np.arange(-124.9375, -66.9375, 0.125)   # 464
CONUS_LAT = np.arange(52.9375, 24.9375, -0.125)     # 224
YEARS     = list(range(2015, 2025))
MONTHS    = list(range(1, 13))

print('Project root:', project_root)
print('CONUS grid:  464 x 224 at 0.125 deg')

## 1. Configuration & Thresholds

In [ ]:
# -- Irrigation threshold -------------------------------------------------------
IRR_THRESHOLD_MM = 20.0   # mm/month: HumanET > 20 = high-confidence irrigation

# -- Cropland masking ----------------------------------------------------------
CDL_CROP_FRAC_MIN = 0.5   # >=50% of cell must be cropland to include

# -- Growing season months -----------------------------------------------------
GROWING_SEASON = [4, 5, 6, 7, 8, 9]   # April-September

# -- Drought categorization from GRIDMET indices -------------------------------
# Following USDM-equivalent thresholds (Svoboda et al. 2002; WMO 2012)
# Category: -1=No drought, 0=D0 (Abnormally Dry), 1=D1, 2=D2, 3=D3, 4=D4
DM_LABELS  = {-1: 'No drought', 0: 'D0', 1: 'D1', 2: 'D2', 3: 'D3', 4: 'D4'}
DM_COLORS  = {-1: '#A8D5A2', 0: '#FFFF00', 1: '#F5BE4E', 2: '#E08C32', 3: '#C74B2A', 4: '#6B0F0F'}

def spei_to_dm(spei):
    """Bin SPEI/SPI values into USDM-equivalent DM categories (-1 to 4)."""
    dm = np.full_like(spei, -1, dtype=float)  # default: no drought
    dm = np.where(spei < -0.5,  0, dm)   # D0
    dm = np.where(spei < -1.0,  1, dm)   # D1
    dm = np.where(spei < -1.5,  2, dm)   # D2
    dm = np.where(spei < -2.0,  3, dm)   # D3
    dm = np.where(spei < -2.5,  4, dm)   # D4
    return dm

def eddi_to_dm(eddi):
    """Bin EDDI values into DM categories. EDDI is positive for dry conditions."""
    dm = np.full_like(eddi, -1, dtype=float)
    dm = np.where(eddi >  0.5,  0, dm)
    dm = np.where(eddi >  1.0,  1, dm)
    dm = np.where(eddi >  1.5,  2, dm)
    dm = np.where(eddi >  2.0,  3, dm)
    dm = np.where(eddi >  2.5,  4, dm)
    return dm

def pdsi_to_dm(pdsi):
    """Bin PDSI into DM categories."""
    dm = np.full_like(pdsi, -1, dtype=float)
    dm = np.where(pdsi < -1.0,  0, dm)
    dm = np.where(pdsi < -2.0,  1, dm)
    dm = np.where(pdsi < -3.0,  2, dm)
    dm = np.where(pdsi < -4.0,  3, dm)
    dm = np.where(pdsi < -5.0,  4, dm)
    return dm

print('Drought thresholds configured.')
print('SPI/SPEI: D0 < -0.5, D1 < -1.0, D2 < -1.5, D3 < -2.0, D4 < -2.5')
print('EDDI:     D0 >  0.5, D1 >  1.0, D2 >  1.5, D3 >  2.0, D4 >  2.5')
print('PDSI:     D0 < -1.0, D1 < -2.0, D2 < -3.0, D3 < -4.0, D4 < -5.0')

## 2. CDL Cropland Mask

Build a spatially explicit cropland mask from CDL fraction rasters. Cells with ≥50% cropland coverage are included in the analysis. Crop-type fractions (corn, soy, wheat) enable crop-specific sub-analyses.

In [ ]:
# Load CDL fraction rasters for each year
# Expected files: cdl_proc / f'CDL_CONUS_{year}_cropland_frac.tif'
#                 cdl_proc / f'CDL_CONUS_{year}_corn_frac.tif'
#                 cdl_proc / f'CDL_CONUS_{year}_soy_frac.tif'

n_lat, n_lon = len(CONUS_LAT), len(CONUS_LON)

cropland_frac_all = np.full((len(YEARS), n_lat, n_lon), np.nan)
corn_frac_all     = np.full((len(YEARS), n_lat, n_lon), np.nan)
soy_frac_all      = np.full((len(YEARS), n_lat, n_lon), np.nan)

cdl_missing = []
for yi, year in enumerate(YEARS):
    fp_crop = cdl_proc / f'CDL_CONUS_{year}_cropland_frac.tif'
    fp_corn = cdl_proc / f'CDL_CONUS_{year}_corn_frac.tif'
    fp_soy  = cdl_proc / f'CDL_CONUS_{year}_soy_frac.tif'

    if fp_crop.exists():
        with rasterio.open(fp_crop) as src:
            arr = src.read(1).astype(float)
            arr[arr == src.nodata] = np.nan if src.nodata is not None else arr[arr]
            # Resize to CONUS grid if needed
            if arr.shape == (n_lat, n_lon):
                cropland_frac_all[yi] = arr
            else:
                print(f'  WARNING: CDL {year} shape mismatch: {arr.shape} vs ({n_lat},{n_lon})')
    else:
        cdl_missing.append(year)

    if fp_corn.exists():
        with rasterio.open(fp_corn) as src:
            arr = src.read(1).astype(float)
            if arr.shape == (n_lat, n_lon):
                corn_frac_all[yi] = arr

    if fp_soy.exists():
        with rasterio.open(fp_soy) as src:
            arr = src.read(1).astype(float)
            if arr.shape == (n_lat, n_lon):
                soy_frac_all[yi] = arr

if cdl_missing:
    print('Missing CDL years:', cdl_missing)

# Mean fraction across all years
cropland_frac_mean = np.nanmean(cropland_frac_all, axis=0)
corn_frac_mean     = np.nanmean(corn_frac_all,     axis=0)
soy_frac_mean      = np.nanmean(soy_frac_all,      axis=0)

# Static cropland mask: >=50% cropland in mean
crop_mask_static = (cropland_frac_mean >= CDL_CROP_FRAC_MIN)

n_crop_cells = crop_mask_static.sum()
n_total      = crop_mask_static.size
print('Static cropland mask (>=50%) covers', n_crop_cells, 'of', n_total, 'CONUS cells')
print('Cropland fraction of CONUS: ' + str(round(100 * n_crop_cells / n_total, 1)) + '%')
print('Mean cropland_frac inside mask:', round(float(np.nanmean(cropland_frac_mean[crop_mask_static])), 3))
print('Mean corn_frac inside mask:    ', round(float(np.nanmean(corn_frac_mean[crop_mask_static])), 3))
print('Mean soy_frac inside mask:     ', round(float(np.nanmean(soy_frac_mean[crop_mask_static])), 3))

In [ ]:
if True:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(18, 5))

        # Panel 1: mean cropland fraction
        ax = axes[0]
        im0 = ax.imshow(
            cropland_frac_mean,
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='YlGn', vmin=0, vmax=1, aspect='auto'
        )
        plt.colorbar(im0, ax=ax, label='Cropland fraction')
        ax.set_title('Mean Cropland Fraction (CDL 2015\u20132024)', fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.contour(
            CONUS_LON, CONUS_LAT, crop_mask_static.astype(float),
            levels=[0.5], colors='k', linewidths=0.5, alpha=0.5
        )

        # Panel 2: corn+soy combined fraction
        ax = axes[1]
        cornsoy = np.nansum([corn_frac_mean, soy_frac_mean], axis=0)
        cornsoy[cornsoy == 0] = np.nan
        im1 = ax.imshow(
            cornsoy,
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='BuGn', vmin=0, vmax=1, aspect='auto'
        )
        plt.colorbar(im1, ax=ax, label='Corn + Soy fraction')
        ax.set_title('Corn + Soy Fraction (CDL 2015\u20132024)', fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.tight_layout()
        plt.savefig(figs / 'conus_cdl_fraction_mean.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_cdl_fraction_mean.png')
    except Exception as e:
        print('CDL map skipped:', e)

## 3. HumanET = OpenET − NLDAS Noah ET

Computes the HumanET signal for each month and grid cell. Positive values indicate anthropogenic water additions (primarily irrigation). Threshold: **>20 mm/month** = high-confidence irrigation.

In [ ]:
# Load all OpenET and NLDAS files, compute delta, mask to cropland
# Store as monthly arrays: shape (n_months, 224, 464)
# n_months = 120 (2015-2024)

records    = []
delta_maps = {}   # key: (year, month) -> 2D array

n_proc = 0
n_fail = 0

for year in YEARS:
    # Per-year cropland mask (use year-specific CDL if available)
    yi = YEARS.index(year)
    year_mask = (cropland_frac_all[yi] >= CDL_CROP_FRAC_MIN)
    if year_mask.sum() == 0:
        year_mask = crop_mask_static   # fallback to static mask

    for month in MONTHS:
        yyyymm = f'{year}{month:02d}'

        fp_openet = openet_proc / f'OpenET_CONUS_{yyyymm}.tif'
        fp_nldas  = nldas_proc  / f'NLDAS_Evap_{yyyymm}.nc'

        openet_arr = None
        nldas_arr  = None

        # Load OpenET
        if fp_openet.exists():
            try:
                with rasterio.open(fp_openet) as src:
                    a = src.read(1).astype(float)
                    nd = src.nodata
                    if nd is not None:
                        a[a == nd] = np.nan
                    if a.shape == (n_lat, n_lon):
                        openet_arr = a
            except Exception as e:
                print(f'  OpenET read error {yyyymm}: {e}')

        # Load NLDAS Noah ET
        if fp_nldas.exists():
            try:
                ds = xr.open_dataset(fp_nldas)
                # Variable name may be 'Evap', 'et', 'EVP', or similar
                et_var = None
                for vname in ['Evap', 'EVP', 'et', 'ET', 'aevap_surface']:
                    if vname in ds:
                        et_var = vname
                        break
                if et_var is None:
                    et_var = list(ds.data_vars)[0]
                arr_n = ds[et_var].values
                ds.close()
                if arr_n.ndim == 3:
                    arr_n = arr_n[0]
                if arr_n.shape == (n_lat, n_lon):
                    nldas_arr = arr_n.astype(float)
                elif arr_n.shape == (n_lon, n_lat):
                    nldas_arr = arr_n.T.astype(float)
            except Exception as e:
                print(f'  NLDAS read error {yyyymm}: {e}')

        if openet_arr is not None and nldas_arr is not None:
            delta = openet_arr - nldas_arr
            delta_masked = np.where(year_mask, delta, np.nan)

            # Mask out extreme outliers (>500 mm/month)
            delta_masked = np.where(np.abs(delta_masked) > 500, np.nan, delta_masked)

            valid     = ~np.isnan(delta_masked)
            irr_cells = (delta_masked > IRR_THRESHOLD_MM) & valid
            n_valid   = valid.sum()
            irr_frac  = irr_cells.sum() / n_valid if n_valid > 0 else np.nan
            mean_d    = float(np.nanmean(delta_masked))

            delta_maps[(year, month)] = delta_masked
            records.append({
                'year':     year,
                'month':    month,
                'yyyymm':   yyyymm,
                'mean_delta': mean_d,
                'irr_frac':   irr_frac,
                'n_cells':    n_valid
            })
            n_proc += 1
        else:
            records.append({
                'year': year, 'month': month, 'yyyymm': yyyymm,
                'mean_delta': np.nan, 'irr_frac': np.nan, 'n_cells': 0
            })
            n_fail += 1

df_et = pd.DataFrame(records)
df_et['date'] = pd.to_datetime(df_et['yyyymm'], format='%Y%m')

print('Months processed:', n_proc)
print('Months missing:  ', n_fail)
print()
print('Mean HumanET by month (growing season):')
gs_df = df_et[df_et['month'].isin(GROWING_SEASON)]
print(gs_df.groupby('month')['mean_delta'].mean().round(2).to_string())

In [ ]:
if True:
    try:
        fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

        # Top: monthly mean HumanET bar chart
        ax = axes[0]
        colors_bar = [
            '#2166ac' if v > IRR_THRESHOLD_MM else '#d1e5f0'
            for v in df_et['mean_delta']
        ]
        ax.bar(df_et['date'], df_et['mean_delta'], color=colors_bar, width=25, align='center')
        ax.axhline(IRR_THRESHOLD_MM, color='red', linestyle='--', linewidth=1,
                   label=f'Irrigation threshold ({IRR_THRESHOLD_MM} mm/mo)')
        ax.axhline(0, color='k', linewidth=0.7)
        ax.set_ylabel('HumanET (mm/month)', fontsize=11)
        ax.set_title('CONUS Cropland HumanET (OpenET \u2212 NLDAS Noah ET), 2015\u20132024', fontsize=13)
        ax.legend(fontsize=9)
        ax.set_ylim(-30, 120)

        # Bottom: fraction of irrigated cropland cells
        ax = axes[1]
        ax.fill_between(df_et['date'], df_et['irr_frac'] * 100, color='#2166ac', alpha=0.7,
                        label='Irrigated fraction')
        ax.set_ylabel('Irrigated cropland cells (%)', fontsize=11)
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylim(0, 100)
        ax.legend(fontsize=9)

        # Mark growing season shading
        for yr in YEARS:
            gs_start = pd.Timestamp(yr, 4, 1)
            gs_end   = pd.Timestamp(yr, 9, 30)
            for ax in axes:
                ax.axvspan(gs_start, gs_end, alpha=0.07, color='green', zorder=0)

        plt.tight_layout()
        plt.savefig(figs / 'conus_human_et_timeseries.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_human_et_timeseries.png')
    except Exception as e:
        print('HumanET timeseries skipped:', e)

In [ ]:
if True:
    try:
        # Compute 10-year mean growing-season HumanET per pixel
        gs_keys  = [(y, m) for y in YEARS for m in GROWING_SEASON if (y, m) in delta_maps]
        gs_stack = np.stack([delta_maps[k] for k in gs_keys], axis=0)
        gs_mean  = np.nanmean(gs_stack, axis=0)

        fig, ax = plt.subplots(figsize=(16, 7))
        vmax = np.nanpercentile(gs_mean[crop_mask_static], 95)
        im = ax.imshow(
            np.where(crop_mask_static, gs_mean, np.nan),
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='Blues', vmin=0, vmax=max(vmax, 1), aspect='auto'
        )
        cbar = plt.colorbar(im, ax=ax, label='HumanET (mm/month)', fraction=0.025, pad=0.02)
        ax.set_title('Mean Growing-Season HumanET, CONUS Cropland (2015\u20132024 Apr\u2013Sep)',
                     fontsize=13)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.tight_layout()
        plt.savefig(figs / 'conus_human_et_map_mean.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_human_et_map_mean.png')
    except Exception as e:
        print('HumanET map skipped:', e)

## 4. SIF — OCO-2 Solar-Induced Fluorescence

Loads harmonized SIF half-monthly files and computes growing-season z-scores by pixel-month using a within-pixel climatology (2015–2024 mean/std per calendar month).

In [ ]:
import re
from glob import glob

sif_files = sorted(sif_proc.glob('SIF_CONUS_*.nc'))
print('SIF files found:', len(sif_files))

# ---------------------------
# Pass 1: accumulate per-month climatology
# sif_clim[month] = list of 2D arrays
# ---------------------------
sif_clim  = {m: [] for m in range(1, 13)}
sif_meta  = []   # list of dicts: yyyymm, half, year, month, mean_sif

for fp in sif_files:
    stem = fp.stem   # e.g. SIF_CONUS_202207a
    m = re.search(r'(\d{6})([ab]?)$', stem)
    if m is None:
        print('  Skipping unrecognized filename:', fp.name)
        continue
    yyyymm = m.group(1)
    half   = m.group(2) if m.group(2) else 'a'
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])

    if month not in GROWING_SEASON:
        continue

    try:
        ds = xr.open_dataset(fp)
        sif_var = None
        for vname in ['sif', 'SIF', 'sif_dc', 'fluorescence']:
            if vname in ds:
                sif_var = vname
                break
        if sif_var is None:
            sif_var = list(ds.data_vars)[0]
        arr = ds[sif_var].values
        ds.close()

        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(float)
        arr[arr < -900] = np.nan
        if arr.shape != (n_lat, n_lon):
            print(f'  SIF shape mismatch {fp.name}: {arr.shape}')
            continue

        sif_clim[month].append(arr)
        sif_meta.append({
            'yyyymm':   yyyymm,
            'half':     half,
            'year':     year,
            'month':    month,
            'mean_sif': float(np.nanmean(arr[crop_mask_static]))
        })
    except Exception as e:
        print(f'  SIF load error {fp.name}: {e}')

# ---------------------------
# Compute pixel-wise climatology per calendar month
# ---------------------------
sif_clim_mean = {}
sif_clim_std  = {}
for mo, arrs in sif_clim.items():
    if len(arrs) > 0:
        stack = np.stack(arrs, axis=0)
        sif_clim_mean[mo] = np.nanmean(stack, axis=0)
        sif_clim_std[mo]  = np.nanstd(stack,  axis=0)

# ---------------------------
# Pass 2: compute z-scores; store aggregate cropland stats
# ---------------------------
sif_zscore_maps = {}   # (yyyymm, half) -> 2D z-score array

for fp in sif_files:
    stem = fp.stem
    m = re.search(r'(\d{6})([ab]?)$', stem)
    if m is None:
        continue
    yyyymm = m.group(1)
    half   = m.group(2) if m.group(2) else 'a'
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])

    if month not in GROWING_SEASON:
        continue
    if month not in sif_clim_mean:
        continue

    try:
        ds = xr.open_dataset(fp)
        sif_var = None
        for vname in ['sif', 'SIF', 'sif_dc', 'fluorescence']:
            if vname in ds:
                sif_var = vname
                break
        if sif_var is None:
            sif_var = list(ds.data_vars)[0]
        arr = ds[sif_var].values
        ds.close()
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(float)
        arr[arr < -900] = np.nan
        if arr.shape != (n_lat, n_lon):
            continue

        mu  = sif_clim_mean[month]
        sig = sif_clim_std[month]
        with np.errstate(invalid='ignore', divide='ignore'):
            z = np.where(sig > 0, (arr - mu) / sig, np.nan)

        sif_zscore_maps[(yyyymm, half)] = z
    except Exception as e:
        print(f'  SIF z-score error {fp.name}: {e}')

df_sif = pd.DataFrame(sif_meta)
if len(df_sif) > 0:
    df_sif['date'] = pd.to_datetime(df_sif['yyyymm'], format='%Y%m')
    print('SIF records (growing season):', len(df_sif))
    print('Mean SIF by month:')
    print(df_sif.groupby('month')['mean_sif'].mean().round(4).to_string())
    print()
    print('Z-score maps computed:', len(sif_zscore_maps))
else:
    print('No SIF files loaded.')

## 5. Drought Categorization

Loads GRIDMET drought indices and USDM categories. Bins SPEI-90d (the 90-day SPEI) into USDM-equivalent DM categories for comparison with SIF anomalies.

In [ ]:
gmet_files = sorted(gmet_proc.glob('GRIDMET_drought_*.tif'))
print('GRIDMET drought files found:', len(gmet_files))

drought_maps = {}   # (year, month) -> dict of band arrays
drought_recs = []

for fp in gmet_files:
    m = re.search(r'(\d{6})', fp.stem)
    if m is None:
        continue
    yyyymm = m.group(1)
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])

    try:
        with rasterio.open(fp) as src:
            n_bands = src.count
            tags    = src.tags()
            bands   = {}
            for b in range(1, n_bands + 1):
                band_tags = src.tags(b)
                bname = band_tags.get('name', band_tags.get('STATISTICS_MINIMUM', f'band{b}'))
                arr = src.read(b).astype(float)
                nd  = src.nodata
                if nd is not None:
                    arr[arr == nd] = np.nan
                bands[f'band{b}'] = arr
                if 'spei90' in bname.lower() or 'spei_90' in bname.lower() or b == 1:
                    bands['spei90d'] = arr
            if 'spei90d' not in bands:
                bands['spei90d'] = bands['band1']

        spei90 = bands['spei90d']
        if spei90.shape == (n_lat, n_lon):
            dm = spei_to_dm(spei90)
            dm_masked = np.where(crop_mask_static, dm, np.nan)
            drought_maps[(year, month)] = {
                'spei90d': spei90,
                'dm_cat':  dm,
                'dm_cat_crop': dm_masked
            }
            drought_recs.append({
                'year':          year,
                'month':         month,
                'yyyymm':        yyyymm,
                'mean_spei90d':  float(np.nanmean(spei90[crop_mask_static])),
                'mean_dm_cat':   float(np.nanmean(dm_masked)),
                'frac_d2plus':   float(np.nanmean(dm_masked >= 2))
            })
    except Exception as e:
        print(f'  GRIDMET error {fp.name}: {e}')

# Load USDM for comparison
usdm_recs = []
usdm_maps = {}
usdm_files = sorted(usdm_proc.glob('USDM_CONUS_*.tif'))
print('USDM files found:', len(usdm_files))

for fp in usdm_files:
    m = re.search(r'(\d{6})', fp.stem)
    if m is None:
        continue
    yyyymm = m.group(1)
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])
    try:
        with rasterio.open(fp) as src:
            arr = src.read(1).astype(float)
            nd  = src.nodata
            if nd is not None:
                arr[arr == nd] = np.nan
        if arr.shape == (n_lat, n_lon):
            dm_crop = np.where(crop_mask_static, arr, np.nan)
            usdm_maps[(year, month)] = arr
            usdm_recs.append({
                'year': year, 'month': month, 'yyyymm': yyyymm,
                'mean_usdm': float(np.nanmean(dm_crop))
            })
    except Exception as e:
        print(f'  USDM error {fp.name}: {e}')

df_drought = pd.DataFrame(drought_recs)
df_usdm    = pd.DataFrame(usdm_recs)

if len(df_drought) > 0 and len(df_usdm) > 0:
    merged = pd.merge(df_drought, df_usdm, on=['year', 'month', 'yyyymm'], how='inner')
    r, p = stats.pearsonr(merged['mean_dm_cat'].dropna(), merged['mean_usdm'].dropna())
    print('Pearson r (SPEI-DM vs USDM): ' + str(round(r, 3)) + '  p=' + str(round(p, 4)))
else:
    print('Insufficient data for SPEI vs USDM comparison.')

print('GRIDMET drought records:', len(drought_recs))
print('USDM records:           ', len(usdm_recs))

In [ ]:
if True:
    try:
        fig, axes = plt.subplots(2, 2, figsize=(18, 10))
        months_plot = [(2022, 7), (2022, 8)]

        dm_cmap  = mcolors.ListedColormap(['#A8D5A2', '#FFFF00', '#F5BE4E', '#E08C32', '#C74B2A', '#6B0F0F'])
        dm_norm  = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5, 3.5, 4.5], dm_cmap.N)
        dm_ticks = [-1, 0, 1, 2, 3, 4]
        dm_tlabs = ['None', 'D0', 'D1', 'D2', 'D3', 'D4']

        for col, (yr, mo) in enumerate(months_plot):
            mo_label = date(yr, mo, 1).strftime('%B %Y')

            # SPEI-derived DM (top row)
            ax = axes[0][col]
            if (yr, mo) in drought_maps:
                dm_arr = drought_maps[(yr, mo)]['dm_cat']
                im = ax.imshow(
                    dm_arr,
                    extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                    origin='upper', cmap=dm_cmap, norm=dm_norm, aspect='auto'
                )
                cbar = plt.colorbar(im, ax=ax, ticks=dm_ticks)
                cbar.ax.set_yticklabels(dm_tlabs)
            ax.set_title(f'SPEI-90d \u2192 DM: {mo_label}', fontsize=11)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')

            # USDM (bottom row)
            ax = axes[1][col]
            if (yr, mo) in usdm_maps:
                im = ax.imshow(
                    usdm_maps[(yr, mo)],
                    extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                    origin='upper', cmap=dm_cmap, norm=dm_norm, aspect='auto'
                )
                cbar = plt.colorbar(im, ax=ax, ticks=dm_ticks)
                cbar.ax.set_yticklabels(dm_tlabs)
            ax.set_title(f'USDM: {mo_label}', fontsize=11)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')

        plt.suptitle('Drought Comparison: SPEI-90d-derived DM vs USDM, Summer 2022',
                     fontsize=13, y=1.01)
        plt.tight_layout()
        plt.savefig(figs / 'conus_drought_comparison_2022.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_drought_comparison_2022.png')
    except Exception as e:
        print('Drought comparison map skipped:', e)

## 6. SIF × Drought — CONUS Cropland

The core analysis: does SIF anomaly track drought severity? Do irrigated crops maintain higher SIF under drought compared to rainfed crops?

This section reproduces Figures 7–12 from the Iowa analysis for all CONUS cropland.

In [ ]:
# Build combined pixel-month DataFrame
# Join SIF z-scores, HumanET (irrigation flag), drought DM category
# For memory management: sample 20% of cropland pixels

np.random.seed(42)
crop_idx = np.argwhere(crop_mask_static)   # (n_cells, 2) row/col indices

# Sample 20% of cropland pixels
sample_frac = 0.20
n_sample    = max(1, int(len(crop_idx) * sample_frac))
sample_sel  = np.random.choice(len(crop_idx), size=n_sample, replace=False)
crop_idx_s  = crop_idx[sample_sel]

rows_s = crop_idx_s[:, 0]
cols_s = crop_idx_s[:, 1]
lats_s = CONUS_LAT[rows_s]
lons_s = CONUS_LON[cols_s]

print(f'Sampled {n_sample:,} of {len(crop_idx):,} cropland pixels ({sample_frac*100:.0f}%)')

combined_rows = []

for year in YEARS:
    for month in GROWING_SEASON:
        yyyymm = f'{year}{month:02d}'

        # HumanET delta
        if (year, month) in delta_maps:
            delta_arr = delta_maps[(year, month)]
        else:
            continue   # skip month if no HumanET

        # Drought DM
        if (year, month) in drought_maps:
            dm_arr     = drought_maps[(year, month)]['dm_cat']
            spei_arr   = drought_maps[(year, month)]['spei90d']
        else:
            dm_arr   = np.full((n_lat, n_lon), np.nan)
            spei_arr = np.full((n_lat, n_lon), np.nan)

        # SIF z-score: prefer 'a' half, fallback to 'b'
        sif_z_arr = None
        for half in ['a', 'b']:
            key = (yyyymm, half)
            if key in sif_zscore_maps:
                sif_z_arr = sif_zscore_maps[key]
                break
        if sif_z_arr is None:
            # Try no suffix
            key = (yyyymm, '')
            if key in sif_zscore_maps:
                sif_z_arr = sif_zscore_maps[key]

        # Extract sampled pixels
        delta_s  = delta_arr[rows_s, cols_s]
        dm_s     = dm_arr[rows_s, cols_s]
        spei_s   = spei_arr[rows_s, cols_s]
        sif_z_s  = sif_z_arr[rows_s, cols_s] if sif_z_arr is not None else np.full(n_sample, np.nan)
        is_irr_s = (delta_s > IRR_THRESHOLD_MM).astype(float)
        is_irr_s[np.isnan(delta_s)] = np.nan

        for i in range(n_sample):
            combined_rows.append({
                'year':        year,
                'month':       month,
                'yyyymm':      yyyymm,
                'lat':         lats_s[i],
                'lon':         lons_s[i],
                'sif_z':       sif_z_s[i],
                'delta_et':    delta_s[i],
                'is_irrigated': is_irr_s[i],
                'dm_cat':      dm_s[i],
                'spei90d':     spei_s[i]
            })

df_combined = pd.DataFrame(combined_rows)
df_combined['date'] = pd.to_datetime(df_combined['yyyymm'], format='%Y%m')

# Drop fully NaN rows
df_combined = df_combined.dropna(subset=['sif_z', 'dm_cat'], how='all')
df_combined['dm_cat_int'] = df_combined['dm_cat'].round().astype('Int64')

print('Combined pixel-month records:', len(df_combined))
print('Columns:', list(df_combined.columns))
print()
if len(df_combined) > 0:
    print('DM category distribution:')
    for c in sorted(df_combined['dm_cat_int'].dropna().unique()):
        n = (df_combined['dm_cat_int'] == c).sum()
        label = DM_LABELS.get(int(c), str(c))
        print('  ' + str(c) + ' (' + label + '): ' + str(n))

In [ ]:
if True:
    try:
        # -- Build monthly CONUS-mean SIF z-score time series --
        if len(df_sif) > 0:
            # Aggregate z-scores from maps
            ts_rows = []
            for (yyyymm, half), z_map in sif_zscore_maps.items():
                mo = int(yyyymm[4:])
                yr = int(yyyymm[:4])
                z_crop = z_map[crop_mask_static]
                ts_rows.append({
                    'year': yr, 'month': mo, 'yyyymm': yyyymm, 'half': half,
                    'mean_sif_z': float(np.nanmean(z_crop))
                })
            df_ts = pd.DataFrame(ts_rows)
            df_ts['date'] = pd.to_datetime(df_ts['yyyymm'], format='%Y%m')
            df_ts = df_ts.sort_values('date')

            # USDM area fractions per month
            usdm_frac_recs = []
            for (yr, mo), usdm_arr in usdm_maps.items():
                u_crop = usdm_arr[crop_mask_static]
                valid  = ~np.isnan(u_crop)
                total  = valid.sum()
                row = {'year': yr, 'month': mo}
                for cat in [0, 1, 2, 3, 4]:
                    row[f'frac_d{cat}'] = (u_crop[valid] >= cat).sum() / total if total > 0 else np.nan
                usdm_frac_recs.append(row)
            df_usdm_frac = pd.DataFrame(usdm_frac_recs)
            df_usdm_frac['date'] = pd.to_datetime(
                df_usdm_frac['year'].astype(str) + df_usdm_frac['month'].astype(str).str.zfill(2),
                format='%Y%m'
            )

            # --- Plot ---
            fig, axes = plt.subplots(3, 1, figsize=(18, 11), sharex=True)

            # Panel 1: SIF z-score time series
            ax = axes[0]
            ax.plot(df_ts['date'], df_ts['mean_sif_z'], color='#2a7b0f', linewidth=1.5, label='CONUS cropland mean SIF z-score')
            ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
            ax.fill_between(df_ts['date'], df_ts['mean_sif_z'], 0,
                            where=df_ts['mean_sif_z'] < 0, alpha=0.3, color='red', label='Negative anomaly')
            ax.fill_between(df_ts['date'], df_ts['mean_sif_z'], 0,
                            where=df_ts['mean_sif_z'] > 0, alpha=0.3, color='green', label='Positive anomaly')
            ax.set_ylabel('SIF z-score', fontsize=11)
            ax.set_title('CONUS Cropland SIF z-score and Drought Severity, 2015\u20132024', fontsize=13)
            ax.legend(fontsize=9, loc='upper left')

            # Panel 2: USDM area fraction stacked
            ax = axes[1]
            if len(df_usdm_frac) > 0:
                df_usdm_frac = df_usdm_frac.sort_values('date')
                cats_plot = [(4, '#6B0F0F', 'D4'), (3, '#C74B2A', 'D3'),
                             (2, '#E08C32', 'D2'), (1, '#F5BE4E', 'D1'), (0, '#FFFF00', 'D0')]
                prev = np.zeros(len(df_usdm_frac))
                for cat, color, label in cats_plot:
                    col_name = f'frac_d{cat}'
                    if col_name in df_usdm_frac.columns:
                        vals = df_usdm_frac[col_name].fillna(0).values
                        ax.bar(df_usdm_frac['date'], vals, bottom=prev,
                               color=color, width=25, label=label, alpha=0.85)
                        prev = prev + vals
            ax.set_ylabel('USDM drought area fraction', fontsize=11)
            ax.set_ylim(0, 1)
            ax.legend(fontsize=9, loc='upper left', ncol=5)

            # Panel 3: SPEI90d CONUS-mean
            ax = axes[2]
            if len(df_drought) > 0:
                df_drought['date'] = pd.to_datetime(df_drought['yyyymm'], format='%Y%m')
                df_d_gs = df_drought[df_drought['month'].isin(GROWING_SEASON)].sort_values('date')
                ax.bar(df_d_gs['date'],
                       df_d_gs['mean_spei90d'],
                       color=df_d_gs['mean_spei90d'].apply(lambda v: '#C74B2A' if v < -1.0 else ('#F5BE4E' if v < 0 else '#A8D5A2')),
                       width=25, alpha=0.8)
                ax.axhline(0,    color='k',   linewidth=0.7)
                ax.axhline(-1.0, color='red', linewidth=0.8, linestyle='--', alpha=0.6, label='D1 threshold')
                ax.axhline(-1.5, color='darkred', linewidth=0.8, linestyle='--', alpha=0.6, label='D2 threshold')
                ax.set_ylabel('Cropland mean SPEI-90d', fontsize=11)
                ax.legend(fontsize=9)
            ax.set_xlabel('Date', fontsize=11)

            plt.tight_layout()
            plt.savefig(figs / 'conus_sif_zscore_timeseries.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_sif_zscore_timeseries.png')
        else:
            print('No SIF data available for timeseries plot.')
    except Exception as e:
        print('SIF timeseries skipped:', e)

In [ ]:
if True:
    try:
        if len(df_combined) == 0 or df_combined['sif_z'].isna().all():
            print('No combined data for violin plot.')
        else:
            import warnings

            df_plot = df_combined.dropna(subset=['sif_z', 'dm_cat_int', 'is_irrigated'])
            present_cats = sorted(df_plot['dm_cat_int'].dropna().unique())
            cat_labels   = [DM_LABELS.get(int(c), str(c)) for c in present_cats]
            cat_colors   = [DM_COLORS.get(int(c), '#aaaaaa') for c in present_cats]

            fig, axes = plt.subplots(1, 2, figsize=(16, 7))
            titles = ['Irrigated (HumanET > 20 mm/mo)', 'Rainfed (HumanET \u2264 20 mm/mo)']
            irr_flags = [1, 0]

            for ax, title, irr_flag in zip(axes, titles, irr_flags):
                sub = df_plot[df_plot['is_irrigated'] == irr_flag]
                if len(sub) == 0:
                    ax.set_title(title + ' (no data)')
                    continue

                groups = [sub[sub['dm_cat_int'] == c]['sif_z'].dropna().values for c in present_cats]
                groups = [g for g in groups if len(g) > 1]

                if len(groups) == 0:
                    ax.set_title(title + ' (insufficient data)')
                    continue

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    parts = ax.violinplot(groups, positions=range(len(groups)),
                                         showmedians=True, showextrema=False)

                for pi, (pc, color) in enumerate(zip(parts['bodies'], cat_colors)):
                    pc.set_facecolor(color)
                    pc.set_alpha(0.75)
                parts['cmedians'].set_color('black')
                parts['cmedians'].set_linewidth(2)

                ax.axhline(0, color='k', linewidth=0.8, linestyle='--', alpha=0.5)
                ax.set_xticks(range(len(groups)))
                ax.set_xticklabels(cat_labels[:len(groups)], fontsize=10)
                ax.set_xlabel('Drought category', fontsize=11)
                ax.set_ylabel('SIF z-score', fontsize=11)
                ax.set_title(title, fontsize=12)

                # Add n labels
                for pi, cat in enumerate(present_cats[:len(groups)]):
                    n = (sub['dm_cat_int'] == cat).sum()
                    ax.text(pi, ax.get_ylim()[0] + 0.05, f'n={n:,}',
                            ha='center', fontsize=7, color='gray')

            plt.suptitle('CONUS Cropland SIF z-score by Drought Category', fontsize=13)
            plt.tight_layout()
            plt.savefig(figs / 'conus_sif_violin_by_drought.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_sif_violin_by_drought.png')
    except Exception as e:
        print('Violin plot skipped:', e)

In [ ]:
if True:
    try:
        if len(df_combined) == 0:
            print('No combined data for ET-by-drought plot.')
        else:
            df_plot = df_combined.dropna(subset=['delta_et', 'dm_cat_int', 'is_irrigated'])
            present_cats = sorted(df_plot['dm_cat_int'].dropna().unique())
            cat_labels   = [DM_LABELS.get(int(c), str(c)) for c in present_cats]

            fig, ax = plt.subplots(figsize=(12, 6))

            x   = np.arange(len(present_cats))
            w   = 0.35

            irr_means = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 1)]['delta_et'].mean()
                         for c in present_cats]
            rai_means = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 0)]['delta_et'].mean()
                         for c in present_cats]
            irr_stds  = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 1)]['delta_et'].std()
                         for c in present_cats]
            rai_stds  = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 0)]['delta_et'].std()
                         for c in present_cats]

            bars_irr = ax.bar(x - w/2, irr_means, width=w, color='#2166ac', alpha=0.8,
                              yerr=irr_stds, capsize=4, label='Irrigated')
            bars_rai = ax.bar(x + w/2, rai_means, width=w, color='#92c5de', alpha=0.8,
                              yerr=rai_stds, capsize=4, label='Rainfed')

            ax.axhline(0, color='k', linewidth=0.7)
            ax.axhline(IRR_THRESHOLD_MM, color='red', linestyle='--', linewidth=1,
                       label=f'Irrigation threshold ({IRR_THRESHOLD_MM} mm/mo)')
            ax.set_xticks(x)
            ax.set_xticklabels(cat_labels, fontsize=11)
            ax.set_xlabel('Drought category (SPEI-90d derived)', fontsize=11)
            ax.set_ylabel('HumanET (mm/month)', fontsize=11)
            ax.set_title('Mean HumanET by Drought Category: Irrigated vs Rainfed Cropland', fontsize=13)
            ax.legend(fontsize=10)

            plt.tight_layout()
            plt.savefig(figs / 'conus_et_by_drought.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_et_by_drought.png')
    except Exception as e:
        print('ET by drought plot skipped:', e)

In [ ]:
if True:
    try:
        # Buffering effect: mean(sif_z | D2+, irrigated) - mean(sif_z | D2+, rainfed) per pixel
        # Work pixel-by-pixel using the full (unsampled) maps to get spatial resolution

        d2plus_months = [(year, month)
                         for year in YEARS for month in GROWING_SEASON
                         if (year, month) in drought_maps and (year, month) in delta_maps]

        irr_sif_accum  = np.zeros((n_lat, n_lon), dtype=float)
        rai_sif_accum  = np.zeros((n_lat, n_lon), dtype=float)
        irr_count      = np.zeros((n_lat, n_lon), dtype=float)
        rai_count      = np.zeros((n_lat, n_lon), dtype=float)

        for (year, month) in d2plus_months:
            yyyymm = f'{year}{month:02d}'
            dm_arr = drought_maps[(year, month)]['dm_cat']
            is_d2plus = (dm_arr >= 2)

            delta_arr = delta_maps[(year, month)]
            is_irr    = (delta_arr > IRR_THRESHOLD_MM)
            is_rai    = (~is_irr) & (~np.isnan(delta_arr))

            sif_z_arr = None
            for half in ['a', 'b', '']:
                key = (yyyymm, half)
                if key in sif_zscore_maps:
                    sif_z_arr = sif_zscore_maps[key]
                    break
            if sif_z_arr is None:
                continue

            valid_sif = ~np.isnan(sif_z_arr)
            in_crop   = crop_mask_static

            mask_irr = is_d2plus & is_irr & valid_sif & in_crop
            mask_rai = is_d2plus & is_rai & valid_sif & in_crop

            irr_sif_accum[mask_irr] += sif_z_arr[mask_irr]
            irr_count[mask_irr]     += 1
            rai_sif_accum[mask_rai] += sif_z_arr[mask_rai]
            rai_count[mask_rai]     += 1

        with np.errstate(invalid='ignore', divide='ignore'):
            irr_mean = np.where(irr_count > 0, irr_sif_accum / irr_count, np.nan)
            rai_mean = np.where(rai_count > 0, rai_sif_accum / rai_count, np.nan)
            buffering_gap = irr_mean - rai_mean

        buffering_gap_crop = np.where(crop_mask_static, buffering_gap, np.nan)

        print('Buffering gap (irrigated - rainfed SIF z-score under D2+):')
        print('  Mean:    ' + str(round(float(np.nanmean(buffering_gap_crop)), 3)))
        print('  Median:  ' + str(round(float(np.nanmedian(buffering_gap_crop)), 3)))
        print('  Std:     ' + str(round(float(np.nanstd(buffering_gap_crop)), 3)))

        # Plot
        fig, ax = plt.subplots(figsize=(16, 7))
        vmax_bg = np.nanpercentile(np.abs(buffering_gap_crop), 95)
        vmax_bg = max(vmax_bg, 0.1)

        im = ax.imshow(
            buffering_gap_crop,
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='RdBu', vmin=-vmax_bg, vmax=vmax_bg, aspect='auto'
        )
        cbar = plt.colorbar(im, ax=ax, label='SIF z-score gap (irrigated \u2212 rainfed)', fraction=0.025)
        ax.set_title('Irrigation Buffering Effect under Drought (D2+): SIF z-score gap\n'
                     'Positive = irrigated crops maintain higher SIF under drought\n'
                     'CONUS Cropland, Growing Season 2015\u20132024', fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.tight_layout()
        plt.savefig(figs / 'conus_sif_buffering_gap_map.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_sif_buffering_gap_map.png')
    except Exception as e:
        print('Buffering gap map skipped:', e)

## 7. Corn vs Soy — SIF and Drought Response

Separate the analysis by dominant crop type (corn vs soy) for cells where corn_frac > 0.5 or soy_frac > 0.5.

In [ ]:
if True:
    try:
        if len(df_combined) == 0:
            print('No combined data for crop-type analysis.')
        else:
            # Add crop type flag based on per-pixel CDL fractions
            df_ct = df_combined.copy()

            # Map lat/lon to row/col
            def latlon_to_rowcol(lat, lon):
                row = np.argmin(np.abs(CONUS_LAT - lat))
                col = np.argmin(np.abs(CONUS_LON - lon))
                return row, col

            corn_frac_pix = corn_frac_mean[rows_s, cols_s]
            soy_frac_pix  = soy_frac_mean[rows_s,  cols_s]

            # Tile across year x month combos
            n_ym = len(YEARS) * len(GROWING_SEASON)
            corn_frac_col = np.tile(corn_frac_pix, n_ym)
            soy_frac_col  = np.tile(soy_frac_pix,  n_ym)

            if len(corn_frac_col) == len(df_ct):
                df_ct['corn_frac'] = corn_frac_col
                df_ct['soy_frac']  = soy_frac_col
            else:
                # Fallback: merge on lat/lon
                pix_df = pd.DataFrame({
                    'lat': lats_s, 'lon': lons_s,
                    'corn_frac': corn_frac_pix,
                    'soy_frac':  soy_frac_pix
                })
                df_ct = df_ct.merge(pix_df, on=['lat', 'lon'], how='left')

            df_ct['crop_type'] = 'other'
            df_ct.loc[df_ct.get('corn_frac', pd.Series(0, index=df_ct.index)) > 0.5, 'crop_type'] = 'corn'
            df_ct.loc[df_ct.get('soy_frac',  pd.Series(0, index=df_ct.index)) > 0.5, 'crop_type'] = 'soy'

            df_ct_gs = df_ct.dropna(subset=['sif_z', 'dm_cat_int'])
            print('Crop type breakdown:')
            for ct in ['corn', 'soy', 'other']:
                n = (df_ct_gs['crop_type'] == ct).sum()
                print('  ' + ct + ': ' + str(n))

            present_cats = sorted(df_ct_gs['dm_cat_int'].dropna().unique())
            cat_labels   = [DM_LABELS.get(int(c), str(c)) for c in present_cats]

            import warnings
            fig, axes = plt.subplots(1, 2, figsize=(16, 7))
            crop_subsets = [('corn', 'Corn-dominant (corn_frac > 0.5)'),
                            ('soy',  'Soy-dominant (soy_frac > 0.5)')]

            for ax, (ctype, ctitle) in zip(axes, crop_subsets):
                sub = df_ct_gs[df_ct_gs['crop_type'] == ctype]
                if len(sub) < 10:
                    ax.set_title(ctitle + ' (insufficient data)')
                    continue

                groups = [sub[sub['dm_cat_int'] == c]['sif_z'].dropna().values for c in present_cats]
                groups_ok = [(c, g) for c, g in zip(present_cats, groups) if len(g) > 1]

                if not groups_ok:
                    ax.set_title(ctitle + ' (insufficient data)')
                    continue

                cats_ok, grps_ok = zip(*groups_ok)

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    parts = ax.violinplot(list(grps_ok), positions=range(len(grps_ok)),
                                         showmedians=True, showextrema=False)

                for pi, (pc, cat) in enumerate(zip(parts['bodies'], cats_ok)):
                    pc.set_facecolor(DM_COLORS.get(int(cat), '#aaaaaa'))
                    pc.set_alpha(0.75)
                parts['cmedians'].set_color('black')
                parts['cmedians'].set_linewidth(2)

                ax.axhline(0, color='k', linewidth=0.8, linestyle='--', alpha=0.5)
                ax.set_xticks(range(len(grps_ok)))
                ax.set_xticklabels([DM_LABELS.get(int(c), str(c)) for c in cats_ok], fontsize=10)
                ax.set_xlabel('Drought category', fontsize=11)
                ax.set_ylabel('SIF z-score', fontsize=11)
                ax.set_title(ctitle, fontsize=12)

            plt.suptitle('SIF z-score by Drought Category: Corn vs Soy\nCONUS Growing Season 2015\u20132024',
                         fontsize=13)
            plt.tight_layout()
            plt.savefig(figs / 'conus_sif_crop_type_drought.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_sif_crop_type_drought.png')
    except Exception as e:
        print('Crop-type panel skipped:', e)

## 8. Summary

In [ ]:
print('=' * 60)
print('CONUS HumanET x SIF x Drought Analysis -- Summary')
print('=' * 60)

# Data coverage
print()
print('--- Data Coverage ---')
print('Years analyzed: 2015-2024 (' + str(len(YEARS)) + ' years)')
et_avail  = df_et[df_et['n_cells'] > 0]
sif_avail = df_sif if len(df_sif) > 0 else pd.DataFrame()
dr_avail  = df_drought if len(df_drought) > 0 else pd.DataFrame()
print('HumanET months available:  ' + str(len(et_avail)) + ' / 120')
print('SIF records (GS only):     ' + str(len(sif_avail)))
print('GRIDMET drought months:    ' + str(len(dr_avail)))
print('USDM months:               ' + str(len(df_usdm)))

# Cropland fraction
print()
print('--- Cropland Coverage ---')
pct_crop = 100 * crop_mask_static.sum() / crop_mask_static.size
print('CONUS cells >=50% cropland: ' + str(crop_mask_static.sum()) +
      ' (' + str(round(pct_crop, 1)) + '% of CONUS grid)')

# Irrigation prevalence
print()
print('--- Irrigation Prevalence ---')
gs_et = df_et[df_et['month'].isin(GROWING_SEASON)]
if len(gs_et) > 0 and gs_et['irr_frac'].notna().any():
    mean_irr_frac = gs_et['irr_frac'].mean()
    print('Mean irrigated cell fraction (GS): ' + str(round(100 * mean_irr_frac, 1)) + '%')
    print('Mean HumanET growing season:       ' + str(round(gs_et['mean_delta'].mean(), 2)) + ' mm/mo')

# SIF anomaly stats by drought category
print()
print('--- SIF z-score by Drought Category ---')
if len(df_combined) > 0 and df_combined['sif_z'].notna().any():
    df_summary = df_combined.dropna(subset=['sif_z', 'dm_cat_int'])
    for c in sorted(df_summary['dm_cat_int'].dropna().unique()):
        sub   = df_summary[df_summary['dm_cat_int'] == c]['sif_z']
        label = DM_LABELS.get(int(c), str(c))
        print('  ' + str(int(c)) + ' (' + label + '): mean=' +
              str(round(sub.mean(), 3)) + '  std=' + str(round(sub.std(), 3)) +
              '  n=' + str(len(sub)))
else:
    print('  No combined SIF-drought data available.')

# Key drought years
print()
print('--- Top 3 Drought Years (cropland-weighted SPEI-90d severity) ---')
if len(df_drought) > 0:
    df_drought_gs = df_drought[df_drought['month'].isin(GROWING_SEASON)].copy()
    if len(df_drought_gs) > 0:
        year_severity = df_drought_gs.groupby('year')['mean_spei90d'].mean().sort_values()
        for yr_rank, (yr, val) in enumerate(year_severity.head(3).items()):
            print('  #' + str(yr_rank+1) + ': ' + str(yr) +
                  '  mean SPEI-90d = ' + str(round(val, 3)))
    else:
        print('  No growing-season drought data.')
else:
    print('  No drought data available.')

print()
print('Figures saved to:', figs)
print('Analysis complete.')